In [ ]:
#-----------------MAIN CODE PROJECT----------------------------------------
import pandas as pd
import random
import time
import os
import matplotlib.pyplot as plt

# PDF Libraries
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image, KeepTogether
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.colors import HexColor

# ✅ SUBJECT-WISE LEADERBOARDS
leaderboards = {
    "Python": [],
    "Fundamental Data Structure": []
}

# ---------------- GRAPH FUNCTION ----------------
def generate_graph(score, total_marks, username):
    labels = ['Score', 'Remaining']
    values = [score, total_marks - score]

    plt.figure()
    plt.pie(values, labels=labels, autopct='%1.1f%%')
    plt.title(f"{username} Performance")

    file_name = f"{username}_graph.png"
    plt.savefig(file_name)
    plt.close()

    return file_name

# ---------------- RESULT PDF FUNCTION ----------------
def generate_pdf(username, quiz_data, user_answers, score, total_marks):
    try:
        file_name = f"{username}_result.pdf"
        doc = SimpleDocTemplate(file_name)

        styles = getSampleStyleSheet()
        content = []

        content.append(Paragraph(f"Quiz Report - {username}", styles['Title']))
        content.append(Spacer(1, 15))

        table_data = [["Question", "Level", "Marks", "Your Answer", "Correct Answer"]]

        for i, q in enumerate(quiz_data):
            question = Paragraph(str(q["QUESTION"]), styles['Normal'])
            level = Paragraph(str(q["LEVEL"]), styles['Normal'])
            marks = Paragraph(str(q["Marks"]), styles['Normal'])
            user_ans = Paragraph(str(user_answers[i]), styles['Normal'])
            correct_ans = Paragraph(str(q["ANSWER"]), styles['Normal'])

            table_data.append([question, level, marks, user_ans, correct_ans])

        table = Table(table_data, colWidths=[220, 60, 60, 100, 120])

        table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), HexColor('#2E86C1')),
            ('TEXTCOLOR', (0,0), (-1,0), colors.white),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),

            ('ALIGN', (1,1), (-1,-1), 'CENTER'),
            ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
            ('GRID', (0,0), (-1,-1), 1, colors.black),

            ('ROWBACKGROUNDS', (0,1), (-1,-1), [
                colors.whitesmoke,
                HexColor('#EBF5FB')
            ]),
        ]))

        content.append(table)
        content.append(Spacer(1, 20))

        percentage = (score / total_marks) * 100 if total_marks > 0 else 0

        content.append(Paragraph(f"Total Score: {score} / {total_marks}", styles['Heading2']))
        content.append(Paragraph(f"Percentage: {round(percentage,2)}%", styles['Heading2']))

        # ✅ GRAPH WITH SAME PAGE HANDLING
        graph_file = generate_graph(score, total_marks, username)

        graph_section = []
        graph_section.append(Spacer(1, 20))
        graph_section.append(Paragraph("Performance Graph", styles['Heading2']))
        graph_section.append(Spacer(1, 10))

        img = Image(graph_file, width=250, height=250) 
        graph_section.append(img)

        content.append(KeepTogether(graph_section))

        doc.build(content)

        print(f"\nPDF Generated Successfully: {file_name}")
        try:
            os.startfile(file_name)
        except:
            pass

    except Exception as e:
        print("PDF Error:", e)


# ---------------- LEADERBOARD PDF FUNCTION ----------------
def generate_leaderboard_pdf(leaderboard, subject):
    try:
        file_name = f"{subject}_Leaderboard.pdf"
        doc = SimpleDocTemplate(file_name)

        styles = getSampleStyleSheet()
        content = []

        content.append(Paragraph(f"{subject} Leaderboard", styles['Title']))
        content.append(Spacer(1, 15))

        table_data = [["Rank", "Name", "Score"]]

        for i, (name, score) in enumerate(leaderboard, 1):
            table_data.append([str(i), name, str(score)])

        table = Table(table_data, colWidths=[80, 200, 100])

        table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), HexColor('#2E86C1')),
            ('TEXTCOLOR', (0,0), (-1,0), colors.white),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),

            ('ALIGN', (0,0), (-1,-1), 'CENTER'),
            ('GRID', (0,0), (-1,-1), 1, colors.black),

            ('ROWBACKGROUNDS', (0,1), (-1,-1), [
                colors.whitesmoke,
                HexColor('#EBF5FB')
            ]),
        ]))

        content.append(table)

        doc.build(content)

        print(f"\n{subject} Leaderboard PDF Generated!")
        try:
            os.startfile(file_name)
        except:
            pass

    except Exception as e:
        print("Leaderboard PDF Error:", e)


# ---------------- LOAD EXCEL ----------------
try:
    data = pd.read_excel("pythonquestion.xlsx")
    data.columns = data.columns.str.strip()
except Exception as e:
    print("Error loading file:", e)
    exit()

print("="*40)
print("WELCOME TO ONLINE QUIZ")
print("="*40)

while True:

    # REGISTER
    print("\nRegister First")
    reg_user = input("Create User Name: ")
    reg_pass = input("Create Password: ")
    print("Registration Successful!!")

    # LOGIN
    while True:
        print("\nLogin")
        username = input("Enter Username: ")
        password = input("Enter Password: ")

        if username == reg_user and password == reg_pass:
            print("Login Successful!!")
            break
        else:
            print("Wrong Username or Password! Try Again.")

    # SUBJECT
    print("\n1. Python\n2. Fundamental Data Structure")
    subject_choice = input("Enter choice: ")

    if subject_choice == "1":
        subject = "Python"
    elif subject_choice == "2":
        subject = "Fundamental Data Structure"
    else:
        print("Invalid choice")
        continue

    # TOTAL MARKS
    try:
        target_marks = int(input("\nEnter total quiz marks: "))
        if target_marks <= 0:
            print("Marks must be positive!")
            continue
    except:
        print("Invalid input!")
        continue

    # FILTER
    filtered = data[data["SUBJECT"].astype(str).str.strip() == subject]

    if filtered.empty:
        print("No questions found!")
        continue

    questions = filtered.to_dict(orient="records")
    random.shuffle(questions)

    quiz = []
    total = 0

    for q in questions:
        if total + q["Marks"] <= target_marks:
            quiz.append(q)
            total += q["Marks"]
        if total == target_marks:
            break

    if total == 0:
        print("Not enough questions!")
        continue

    score = 0
    user_answers = []

    print("\nQuiz Started (10 sec per question)\n")

    # QUIZ
    for q in quiz:
        options = [
            ("a", str(q["OPTIONA"]).strip()),
            ("b", str(q["OPTIONB"]).strip()),
            ("c", str(q["OPTIONC"]).strip()),
            ("d", str(q["OPTIOND"]).strip())
        ]

        # ✅ FIXED LOGIC
        correct_text = str(q["ANSWER"]).strip().lower()
        correct_answer = None

        for key, value in options:
            if correct_text == value.lower():
                correct_answer = key
                break

        print("\n" + q["QUESTION"])
        for key, value in options:
            print(f"{key}) {value}")

        start = time.time()
        ans = input("Enter Answer (a/b/c/d): ").lower().strip()
        end = time.time()

        if end - start > 30:
            print("Time's up!")
            ans = ""

        selected_text = "Not Answered"
        for key, value in options:
            if key == ans:
                selected_text = value

        user_answers.append(selected_text)

        if ans == correct_answer:
            score += q["Marks"]

    # RESULT
    total_marks = sum(q["Marks"] for q in quiz)
    percentage = (score / total_marks) * 100 if total_marks > 0 else 0

    print("="*40)
    print("Score:", score, "/", total_marks)
    print("Percentage:", round(percentage, 2), "%")
    print("PASS" if percentage >= 50 else "FAIL")

    # LEADERBOARD
    leaderboards[subject].append((username, score))
    leaderboards[subject].sort(key=lambda x: x[1], reverse=True)

    print(f"\n{subject} LEADERBOARD")
    for i, (name, sc) in enumerate(leaderboards[subject], 1):
        medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else ""
        print(i, medal, name, "-", sc)

    # PDFs
    generate_pdf(username, quiz, user_answers, score, total_marks)
    generate_leaderboard_pdf(leaderboards[subject], subject)

    again = input("\nNext student? (yes/no): ").lower()
    if again != "yes":
        print("Quiz Ended")
        break

ERROR! Session/line number was not unique in database. History logging moved to new session 89
WELCOME TO ONLINE QUIZ

Register First


Create User Name:  aarti
Create Password:  123


Registration Successful!!

Login


Enter Username:  aarti
Enter Password:  123


Login Successful!!

1. Python
2. Fundamental Data Structure


...


Which data structure does not allow duplicate values?
a) list
b) Dictionary
c) Tuple
d) Set


Enter Answer (a/b/c/d):  b



What will be the output of print(5//2)?
a) 2.5
b) 3
c) 2
d) Error


Enter Answer (a/b/c/d):  c


Score: 3 / 10
Percentage: 30.0 %
FAIL

Python LEADERBOARD
1 🥇 aarti - 3

PDF Generated Successfully: aarti_result.pdf

Python Leaderboard PDF Generated!
